In [ ]:
# ==========================================
# CELL 1: SETUP & DATASET CHECK
# ==========================================
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

# ── Configuration ───────────────────────────────────────
IMAGE_FOLDER     = r"/kaggle/input/datasets/deviprasad007/fucking2ndtime/Ready_for_traning"
CSV_FILE         = r"/kaggle/input/datasets/deviprasad007/entrasuribabu/metadata_embed.csv"
OUTPUT_FEATURES  = r"/kaggle/working/features.npy"
OUTPUT_FILENAMES = r"/kaggle/working/filenames.npy"

IMAGE_SIZE  = 1024
BATCH_SIZE  = 64
NUM_WORKERS = 2
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to: {DEVICE}")

# ── Laterality normalization map ─────────────────────────
LAT_MAP = {'L': 'L', 'LEFT': 'L', 'R': 'R', 'RIGHT': 'R'}

# ── Dataset ──────────────────────────────────────────────
class SmartMammogramDataset(Dataset):
    def __init__(self, csv_file, folder, transform=None):
        print("Building laterality map from CSV...")
        df = pd.read_csv(csv_file, low_memory=False)

        # Vectorized build — much faster than iterrows()
        df['base_id']   = df['filename'].astype(str).str.replace('.dcm', '', regex=False)
        df['lat_clean'] = (df['laterality'].astype(str)
                           .str.strip().str.upper()
                           .map(LAT_MAP).fillna('UNKNOWN'))
        self.lat_dict = dict(zip(df['base_id'], df['lat_clean']))

        # Coverage report
        counts = pd.Series(self.lat_dict.values()).value_counts().to_dict()
        print(f"  Laterality breakdown: {counts}")

        print("Scanning directories for PNGs (both batches)...")
        # os.walk recurses into batch_1 AND batch_2 automatically
        self.paths = sorted(
            os.path.join(root, f)
            for root, _, files in os.walk(folder)
            for f in files if f.lower().endswith('.png')   # case-insensitive
        )
        self.transform = transform
        print(f"✅ {len(self.lat_dict)} CSV records | {len(self.paths)} PNG files found")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]

        # Safe base_id extraction using splitext
        base_id    = os.path.splitext(os.path.basename(img_path))[0]
        laterality = self.lat_dict.get(base_id, 'UNKNOWN')

        img = Image.open(img_path).convert('RGB')

        # Flip right-breast images so all images face left (alignment protocol)
        if laterality == 'R':
            img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)  # Pillow 10+ safe

        if self.transform:
            img = self.transform(img)
        return img, img_path

# ── Transforms ───────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

dataset = SmartMammogramDataset(CSV_FILE, IMAGE_FOLDER, transform=transform)
print(f" Dataset ready: {len(dataset)} images")

In [ ]:
# ==========================================
# CELL 2: MODEL INITIALIZATION
# ==========================================
print("Loading EfficientNet-B4 (Ensure Kaggle Internet is ON)...")

backbone = models.efficientnet_b4(
    weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1
)

# nn.Flatten guarantees (B, 1792) output regardless of torchvision version
# nn.Identity() can leave shape as (B, 1792, 1, 1) in some versions
backbone.classifier = nn.Flatten(start_dim=1)

feature_extractor = backbone.to(DEVICE)

# Multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Firing up {torch.cuda.device_count()} GPUs with DataParallel!")
    feature_extractor = nn.DataParallel(feature_extractor)

feature_extractor.eval()
print(" EfficientNet-B4 feature extractor ready.")

In [ ]:
# ==========================================
# CELL 3: EXTRACT AND SAVE
# ==========================================
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),       # faster host→GPU transfers
    persistent_workers=(NUM_WORKERS > 0),     # avoids re-spawning workers each batch
)

all_features  = []
all_filenames = []

print(f"Total images : {len(dataset)}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Total batches: {len(loader)}")
print("Starting extraction...\n")

# inference_mode is faster than no_grad — disables full autograd engine
with torch.inference_mode(), torch.autocast(device_type=DEVICE.type):
    for imgs, paths in tqdm(loader, desc="Extracting Features", unit="batch"):
        imgs  = imgs.to(DEVICE)
        feats = feature_extractor(imgs)
        all_features.append(feats.cpu().float().numpy())  # .float() converts fp16→fp32
        all_filenames.extend(paths)

# Stack and save
all_features = np.vstack(all_features)
np.save(OUTPUT_FEATURES,  all_features)
np.save(OUTPUT_FILENAMES, np.array(all_filenames))

print(f"\n Done!")
print(f"Feature matrix shape : {all_features.shape}")
print(f"Saved features to    : {OUTPUT_FEATURES}")
print(f"Saved filenames to   : {OUTPUT_FILENAMES}")